In [87]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

pio.templates.default = "plotly_dark"

In [88]:
df = pd.read_csv("Food_Delivery_Times.csv")

display(df.info())
display(df.describe())
display(df.head(10))
print(df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                1000 non-null   int64  
 1   Distance_km             1000 non-null   float64
 2   Weather                 970 non-null    str    
 3   Traffic_Level           970 non-null    str    
 4   Time_of_Day             970 non-null    str    
 5   Vehicle_Type            1000 non-null   str    
 6   Preparation_Time_min    1000 non-null   int64  
 7   Courier_Experience_yrs  970 non-null    float64
 8   Delivery_Time_min       1000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 91.3 KB


None

,Order_ID,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
count,1000.000000,1000.000000,1000.000000,970.000000,1000.000000
mean,500.500000,10.059970,16.982000,4.579381,56.732000
std,288.819436,5.696656,7.204553,2.914394,22.070915
min,1.000000,0.590000,5.000000,0.000000,8.000000
25%,250.750000,5.105000,11.000000,2.000000,41.000000
50%,500.500000,10.190000,17.000000,5.000000,55.500000
75%,750.250000,15.017500,23.000000,7.000000,71.000000
max,1000.000000,19.990000,29.000000,9.000000,153.000000


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68
5,679,19.40,Clear,Low,Evening,Scooter,8,9.0,57
6,627,9.52,Clear,Low,NaN,Bike,12,1.0,49
7,514,17.39,Clear,Medium,Evening,Scooter,5,6.0,46
8,860,1.78,Snowy,Low,Evening,Car,20,6.0,35
9,137,10.62,Foggy,Low,Evening,Scooter,29,1.0,73


0


In [89]:
print(df["Traffic_Level"].unique())
print(df["Time_of_Day"].unique())
print(df["Vehicle_Type"].unique())

<ArrowStringArray>
['Low', 'Medium', 'High', nan]
Length: 4, dtype: str
<ArrowStringArray>
['Afternoon', 'Evening', 'Night', 'Morning', nan]
Length: 5, dtype: str
<ArrowStringArray>
['Scooter', 'Bike', 'Car']
Length: 3, dtype: str


In [90]:
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].mode()[0])
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].astype(int)

In [91]:
display(df["Courier_Experience_yrs"].info())
display(df["Courier_Experience_yrs"].describe())

<class 'pandas.Series'>
RangeIndex: 1000 entries, 0 to 999
Series name: Courier_Experience_yrs
Non-Null Count  Dtype
--------------  -----
1000 non-null   int64
dtypes: int64(1)
memory usage: 7.9 KB


None

count    1000.000000
mean        4.622000
std         2.880523
min         0.000000
25%         2.000000
50%         5.000000
75%         7.000000
max         9.000000
Name: Courier_Experience_yrs, dtype: float64

In [92]:
fig = px.pie(df, names='Weather', title='Weather Distribution')
fig.show()

In [93]:
fig = px.histogram(df, x='Delivery_Time_min', nbins=30, title='Delivery Time Distribution')
fig.show()



In [94]:
display(df.loc[df["Delivery_Time_min"] >= 140])

df = df.drop(df.loc[df["Delivery_Time_min"] >= 140].index).reset_index(drop=True)
display(df.describe())

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
127,446,18.97,Clear,Low,Evening,Car,25,4,141
379,814,18.46,Clear,NaN,NaN,Scooter,29,1,153
452,394,15.64,Rainy,Low,NaN,Bike,20,4,141


,Order_ID,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
count,997.000000,997.000000,997.000000,997.000000,997.000000
mean,500.347041,10.037011,16.958877,4.626881,56.466399
std,289.058560,5.689226,7.200186,2.882435,21.562884
min,1.000000,0.590000,5.000000,0.000000,8.000000
25%,250.000000,5.030000,11.000000,2.000000,41.000000
50%,501.000000,10.160000,17.000000,5.000000,55.000000
75%,750.000000,14.940000,23.000000,7.000000,71.000000
max,1000.000000,19.990000,29.000000,9.000000,126.000000


In [95]:
fig = px.scatter(df, x="Distance_km", y="Delivery_Time_min", title="Delivery Time vs Distance by Traffic Level")
fig.show()

In [96]:
fig = px.scatter(df, x="Delivery_Time_min", y="Preparation_Time_min", title="Delivery Time vs Preparation Time (min)", trendline="ols")
fig.update_traces(line_color="red")
fig.show()

In [97]:
weather_df = df[["Weather", "Delivery_Time_min"]]

weather_df.sort_values(by="Delivery_Time_min")

,Weather,Delivery_Time_min
43,Clear,8
241,Clear,13
95,Clear,14
276,Clear,14
322,Clear,14
...,...,...
547,Clear,116
139,Foggy,116
921,Windy,122
29,Clear,123


In [100]:
weather = px.violin(weather_df, x="Weather", y="Delivery_Time_min", title="Weather vs Delivery Time", points="outliers")
traffic = px.violin(df, x="Traffic_Level", y="Delivery_Time_min", title="Traffic Level vs Delivery Time", points="outliers")
tod = px.box(df, x="Time_of_Day", y="Delivery_Time_min", title="Time of Day vs Delivery Time")
vehicle = px.box(df, x="Vehicle_Type", y="Delivery_Time_min", title="Vehicle Type vs Delivery Time")
experience = px.box(df, x="Courier_Experience_yrs", y="Delivery_Time_min", title="Courier Experience vs Delivery Time")


weather.update_traces(meanline_visible=True)
traffic.update_traces(meanline_visible=True)
weather.show()
traffic.show()
tod.show()
vehicle.show()
experience.show()

In [66]:
print(df.groupby("Traffic_Level")["Delivery_Time_min"].agg(["mean", "median"]))
print(df.groupby("Weather")["Delivery_Time_min"].agg(["mean", "median"]))
print(df.groupby("Time_of_Day")["Delivery_Time_min"].agg(["mean", "median"]))
print(df.groupby("Vehicle_Type")["Delivery_Time_min"].agg(["mean", "median"]))

                    mean  median
Traffic_Level                   
High           64.807107    65.0
Low            52.422572    51.0
Medium         56.020513    53.5
              mean  median
Weather                   
Clear    52.681624    52.0
Foggy    59.466019    59.0
Rainy    59.394089    57.0
Snowy    67.113402    66.0
Windy    55.458333    55.0
                  mean  median
Time_of_Day                   
Afternoon    56.080986    56.0
Evening      57.195205    55.5
Morning      56.120130    56.0
Night        55.211765    52.0
                   mean  median
Vehicle_Type                   
Bike          56.406375    56.0
Car           57.773196    56.0
Scooter       55.724252    54.0


In [67]:
correlation_matrix = df.corr(numeric_only=True)
fig = px.imshow(correlation_matrix, text_auto=True, title="Correlation Matrix", aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.show()

In [68]:
# filling NaN of categorical values

# Traffic level
df.loc[df["Traffic_Level"].isna(), "Traffic_Level"] = df["Traffic_Level"].mode()[0]

# Weather
df.loc[df["Weather"].isna(), "Weather"] = df["Weather"].mode()[0]

# Time of day
df.loc[df["Time_of_Day"].isna(), "Time_of_Day"] = df["Time_of_Day"].mode()[0]

# df.isna().sum()

In [69]:
# Weather, Time of day, vehicle type -> one hot encoding
weather_df = pd.get_dummies(df["Weather"])
tod_df = pd.get_dummies(df["Time_of_Day"])
vehicle_df = pd.get_dummies(df["Vehicle_Type"])

# traffic level -> ordinal encoding
traffic_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

traffic_encoded = df["Traffic_Level"].map(traffic_map)
df["Traffic_Enc"] = traffic_encoded
df = df.join(weather_df)
df = df.join(tod_df)
df = df.join(vehicle_df)

df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min,Traffic_Enc,...,Rainy,Snowy,Windy,Afternoon,Evening,Morning,Night,Bike,Car,Scooter
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1,43,1,...,False,False,True,True,False,False,False,False,False,True
1,738,16.42,Clear,Medium,Evening,Bike,20,2,84,2,...,False,False,False,False,True,False,False,True,False,False
2,741,9.52,Foggy,Low,Night,Scooter,28,1,59,1,...,False,False,False,False,False,False,True,False,False,True
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1,37,2,...,True,False,False,True,False,False,False,False,False,True
4,412,19.03,Clear,Low,Morning,Bike,16,5,68,1,...,False,False,False,False,False,True,False,True,False,False


In [70]:
df = df.drop(["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type", "Order_ID"], axis=1)

In [71]:
correlation_matrix = df.corr(numeric_only=True, method="pearson")
fig = px.imshow(correlation_matrix, text_auto=True, title="Correlation Matrix", aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.show()

In [72]:
display(df.head())
df.info()

,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min,Traffic_Enc,Clear,Foggy,Rainy,Snowy,Windy,Afternoon,Evening,Morning,Night,Bike,Car,Scooter
0,7.93,12,1,43,1,False,False,False,False,True,True,False,False,False,False,False,True
1,16.42,20,2,84,2,True,False,False,False,False,False,True,False,False,True,False,False
2,9.52,28,1,59,1,False,True,False,False,False,False,False,False,True,False,False,True
3,7.44,5,1,37,2,False,False,True,False,False,True,False,False,False,False,False,True
4,19.03,16,5,68,1,True,False,False,False,False,False,False,True,False,True,False,False


<class 'pandas.DataFrame'>
RangeIndex: 997 entries, 0 to 996
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Distance_km             997 non-null    float64
 1   Preparation_Time_min    997 non-null    int64  
 2   Courier_Experience_yrs  997 non-null    int64  
 3   Delivery_Time_min       997 non-null    int64  
 4   Traffic_Enc             997 non-null    int64  
 5   Clear                   997 non-null    bool   
 6   Foggy                   997 non-null    bool   
 7   Rainy                   997 non-null    bool   
 8   Snowy                   997 non-null    bool   
 9   Windy                   997 non-null    bool   
 10  Afternoon               997 non-null    bool   
 11  Evening                 997 non-null    bool   
 12  Morning                 997 non-null    bool   
 13  Night                   997 non-null    bool   
 14  Bike                    997 non-null    bool   
 15  

In [73]:
X = df.drop(columns="Delivery_Time_min")
y = df["Delivery_Time_min"]

display(X.head())
display(y.head())

,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Traffic_Enc,Clear,Foggy,Rainy,Snowy,Windy,Afternoon,Evening,Morning,Night,Bike,Car,Scooter
0,7.93,12,1,1,False,False,False,False,True,True,False,False,False,False,False,True
1,16.42,20,2,2,True,False,False,False,False,False,True,False,False,True,False,False
2,9.52,28,1,1,False,True,False,False,False,False,False,False,True,False,False,True
3,7.44,5,1,2,False,False,True,False,False,True,False,False,False,False,False,True
4,19.03,16,5,1,True,False,False,False,False,False,False,True,False,True,False,False


0    43
1    84
2    59
3    37
4    68
Name: Delivery_Time_min, dtype: int64

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = .2,
    random_state=42
)

model = RandomForestRegressor()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the numb

In [75]:
predictions = model.predict(X_test)

In [76]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

In [77]:
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 8.1457
RMSE: 12.623398908376458
R2: 0.6564173255361978


In [78]:
fi = pd.DataFrame(data={
        "features": X.columns,
        "values": model.feature_importances_
    }
)
fi = fi.sort_values(by=["values"], ascending=False)
print(fi)

fig = px.histogram(fi, x="features", y="values")
fig.show()

                  features    values
0              Distance_km  0.718409
1     Preparation_Time_min  0.134742
3              Traffic_Enc  0.037823
2   Courier_Experience_yrs  0.032872
4                    Clear  0.014210
7                    Snowy  0.010024
5                    Foggy  0.007165
14                     Car  0.006364
10                 Evening  0.006113
6                    Rainy  0.005598
13                    Bike  0.005261
11                 Morning  0.005080
15                 Scooter  0.004882
9                Afternoon  0.004882
12                   Night  0.004627
8                    Windy  0.001947


In [79]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

# scorings = [
#     "neg_mean_absolute_error",
#     "neg_root_mean_squared_error",
#     "r2"
# ]

# for name, model in models.items():
#     print(name.upper())
#     for metric in scorings:
#         scores = cross_val_score(
#             model,
#             X, y,
#             cv=5,
#             scoring=metric
#         )

#         print("*" * 5, metric, "*" * 5)
#         print(f"scores: {scores}")
#         print(f"scores.mean(): {scores.mean():.2f}")
#         print(f"STD: {scores.std():.2f}")
#         print("*" * 10)

model = LinearRegression()
model.fit(X_train, y_train)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"{mae:.2f}; {rmse:.2f}; {r2:.2f}")
joblib.dump(model, "LineRegress.pkl")

8.15; 12.62; 0.66


['LineRegress.pkl']

In [80]:
# df = df.drop(columns=["Clear", "Foggy", "Rainy", "Snowy", "Windy", "Afternoon", "Evening", "Morning", "Night", "Bike", "Car", "Scooter"])
# df.head()

# X = df.drop(columns="Delivery_Time_min")
# y = df["Delivery_Time_min"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=.2,
#     random_state=42
# )

# model = RandomForestRegressor()
# model.fit(X_train, y_train)

# prediction = model.predict(X_test)

# mae = mean_absolute_error(y_test, predictions)
# rmse = np.sqrt(mean_squared_error(y_test, predictions))
# r2 = r2_score(y_test, predictions)

# print(mae)
# print(rmse)
# print(r2)

# fi = pd.DataFrame(data={
#         "features": X.columns,
#         "values": model.feature_importances_
#     }
# )
# fi = fi.sort_values(by=["values"], ascending=False)
# print(fi)

# fig = px.histogram(fi, x="features", y="values")
# fig.show()

In [81]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

metrics = pd.DataFrame(columns=["name", "r2", "mae", "rmse"])

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(np.mean((y_test - predictions)**2))
    r2 = r2_score(y_test, predictions)

    metrics.loc[len(metrics)] = [name, round(r2, 2), round(mae, 2), round(rmse, 2)]

    print(f"{name}:")
    print(f"mae: {mae}")
    print(f"rmse: {rmse}")
    print(f"r2: {r2}")
    print("-" * 30)

Linear Regression:
mae: 7.388858778912154
rmse: 12.0028788019444
r2: 0.6893656454429266
------------------------------
Decision Tree Regressor:
mae: 11.27
rmse: 16.578600664712326
r2: 0.40738261968685285
------------------------------
Random Forest Regressor:
mae: 8.328899999999999
rmse: 12.824294171610381
r2: 0.6453944029299038
------------------------------
Gradient Boosting Regressor:
mae: 8.255342476530823
rmse: 12.635450943414916
r2: 0.6557609496767477
------------------------------


In [82]:
fig = go.Figure(data=[
    go.Bar(name="MAE", x=list(models.keys()), y=list(metrics["mae"]), text=list(metrics["mae"])),
    go.Bar(name="RMSE", x=list(models.keys()), y=list(metrics["rmse"]), text=list(metrics["rmse"])),
])

fig.update_layout(barmode="group")
fig.show()

metrics = metrics.sort_values(by="r2", ascending=False)
fig = px.bar(metrics, x="name", y="r2", text_auto=True)
fig.show()

In [83]:
scorings = [
    "neg_mean_absolute_error",
    "neg_root_mean_squared_error",
    "r2"
]

for name, model in models.items():
    print(name.upper())
    for metric in scorings:
        scores = cross_val_score(
            model,
            X, y,
            cv=5,
            scoring=metric
        )

        print("*" * 5, metric, "*" * 5)
        print(f"scores: {scores}")
        print(f"scores.mean(): {scores.mean():.2f}")
        print(f"STD: {scores.std():.2f}")
        print("*" * 10)

model = LinearRegression()
model.fit(X_train, y_train)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"{mae:.2f}; {rmse:.2f}; {r2:.2f}")

LINEAR REGRESSION
***** neg_mean_absolute_error *****
scores: [-7.0471608  -5.75321848 -6.20833278 -6.62574005 -6.48253475]
scores.mean(): -6.42
STD: 0.43
**********
***** neg_root_mean_squared_error *****
scores: [-11.8787996   -8.63117784  -9.04695944 -10.75094317  -9.6680571 ]
scores.mean(): -10.00
STD: 1.18
**********
***** r2 *****
scores: [0.70224539 0.84065253 0.82100237 0.75235962 0.79299325]
scores.mean(): 0.78
STD: 0.05
**********
DECISION TREE REGRESSOR
***** neg_mean_absolute_error *****
scores: [-11.28        -9.85       -10.64824121 -12.75879397 -11.38190955]
scores.mean(): -11.18
STD: 0.96
**********
***** neg_root_mean_squared_error *****
scores: [-16.97291961 -14.66356028 -15.29853673 -18.08758701 -16.18004851]
scores.mean(): -16.24
STD: 1.21
**********
***** r2 *****
scores: [0.39210791 0.54007831 0.48815109 0.29904587 0.42021666]
scores.mean(): 0.43
STD: 0.08
**********
RANDOM FOREST REGRESSOR
***** neg_mean_absolute_error *****
scores: [-8.1791     -7.2436     -7.25

In [108]:
fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Model</b>",'<b>MAE</b>','<b>RMSE</b>','<b>R2</b>'],
        line_color="darkslategray",
        align=["left", "center"],
        font=dict(size=16)
    ),
    cells=dict(
        values=[
            ["Linear Regression", "Gradient Boosting", "Random Forest Regression", "Decision Tree"],
            ["6.42", "7.09", "7.52", "11.18"],
            ["10", "10.69", "11.06", "16.24"],
            ["0.78", "0.75", "0.73", "0.43"],
        
        ],
        align=["left", "center"],
        fill_color = [["#34d399", "#506784"]],
        font = dict(size=14)
    )
)])

fig.show()

In [85]:
mae = [6.42, 11.18, 7.52, 7.09]
rmse = [10, 10.69, 11.06, 16.24]
r2 = [0.78, 0.75, 0.73, 0.43]

metrics["r2"] = r2
print(metrics["r2"])

fig = go.Figure(data=[
    go.Bar(name="MAE", x=list(models.keys()), y=mae, text=mae),
    go.Bar(name="RMSE", x=list(models.keys()), y=rmse, text=rmse),
])

fig.update_layout(barmode="group")
fig.show()

metrics = metrics.sort_values(by="r2", ascending=False)
fig = px.bar(metrics, x="name", y="r2", text_auto=True)
fig.show()

0    0.78
3    0.75
2    0.73
1    0.43
Name: r2, dtype: float64
